# Select and characterize 3216 pig gut metagenomes

In [1]:
import polars as pl

In [3]:
# remove shitty entries with shitty metatada
unnannotated_values = [
    "metagenome",
    "gut metagenome",
    "feces metagenome",
    "manure metagenome",
    "bacterium",
    "unidentified",
    "null"
    ]
def remove_unnannotated(df: pl.DataFrame) -> pl.DataFrame:
    return df.filter(~pl.col("organism").is_in(unnannotated_values))


# group by organism and count number of bacteria per SRA 
def pivot_count(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .group_by(["broad_cat", "count"])
        .len()  # count occurrences
        .pivot(
            values="len",
            index="broad_cat",
            on="count"
        )
        .fill_null(0)  # optional: fill missing with 0
    )

# set mapping for "categories of interest" I like human v pig v neither. 
category_map_simple = {
    'human associated': ['human', 'homo', 'sapiens'],
    'pig': ['pig', 'sus', 'scrofa']}

# Function to assign category based on keywords above
def get_broad_cat_simple(organism):
    for cat, keywords in category_map_simple.items():
        if any(keyword.lower() in str(organism).lower() for keyword in keywords):
            return cat
    return 'other'  


In [4]:
# get metadata, only compare WGS
metadata_df = (
    pl.scan_parquet("/group/ctbrowngrp5/sra-metagenomes/20241128-metadata.parquet")
    .filter(pl.col("acc") != "NP")
    .filter(pl.col("assay_type") == "WGS")
#    .select(["acc", "organism", "bioproject", "mbases", "host", "project_name"]) 
    .collect()
)
print(f"found metadata records for {len(metadata_df)} WGS metagenome data sets from the SRA")

found metadata records for 703916 WGS metagenome data sets from the SRA


### Create a new dataframe that will provide a 'simpleorg' column: human, pig, other

In [5]:
rename_df = []
for organism in metadata_df['organism'].unique().to_list():
    simpleorg = 'other'
    if organism:
        for kw in ['human', 'homo']:
            if kw in organism.lower():
                simpleorg = 'human'
                break
        for kw in ['pig', 'sus', 'scrofa']:
            if kw in organism.lower():
                simpleorg = 'pig'
                break
    rename_df.append(dict(organism=organism, broad_cat=simpleorg))

rename_df = pl.DataFrame(rename_df)
rename_df['broad_cat'].value_counts().sort(by='count', descending=True)

broad_cat,count
str,u32
"""other""",10208
"""human""",165
"""pig""",36


### Apply the renaming to the metadata table => metadata2_df

In [6]:
metadata2_df = metadata_df.join(rename_df, on='organism', how='inner', coalesce=True)
assert len(metadata2_df) == len(metadata_df)
metadata2_df

sample_name,sample_name_sam,acc,assay_type,avgspotlen,bioproject,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search,broad_cat
str,list[str],str,str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""27""",[],"""SRR28523869""","""WGS""",5336,"""PRJNA1095378""","""SAMN40717046""","[""MIMARKS.survey"", ""MIGS/MIMS/MIMARKS.human-associated""]","""UNIVERSITY OF FLORIDA""",2019-01-21,"""public""","[""fastq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX24124678""","""USA""","""North America""","[""USA""]",null,"""MinION""","""27""","""SINGLE""","""RANDOM""","""METAGENOMIC""",null,258,221,"""human metagenome""","""OXFORD_NANOPORE""",2024-04-02 00:00:00 UTC,"""SRS20910198""","""SRP499205""",null,null,null,null,"""[""2019-01-21""]""",null,null,"""[""human stool""]""",null,"""[""preterm neonate stool""]""",null,"""[""not applicable""]""",null,"""[""Homo sapiens""]""",null,null,null,null,null,null,null,null,null,"""""29.6520 N 82.3250 W""""",null,null,null,null,"""258178589""","""232696690""","""""2024-04-02T09:18:00.000Z""""",null,"""""1095378""""","""human"""
"""CN20""",[],"""SRR19901137""","""WGS""",709,"""PRJNA854064""","""SAMN29417614""","[""Pathogen.cl""]","""ACADEMY OF MILITARY MEDICAL SC…",null,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX15943916""","""China""","""Asia""","[""China""]",null,"""MinION""","""CN20""","""SINGLE""","""RANDOM PCR""","""METAGENOMIC""",null,40,35,"""Streptococcus suis""","""OXFORD_NANOPORE""",2022-06-29 00:00:00 UTC,"""SRS13630891""","""SRP384224""",null,null,null,null,"""[""2020""]""",null,null,null,null,null,null,null,null,"""[""Homo sapiens""]""",null,null,null,null,null,null,null,null,null,"""""39.56 N 116.246 E""""",null,null,null,null,"""40695094""","""37326224""","""""2022-06-29T14:47:00.000Z""""",null,"""""29417614""""","""other"""
"""9419-17""",[],"""SRR24962475""","""WGS""",2830,"""PRJNA982891""","""SAMN35766867""","[""Pathogen.cl""]","""UNIVERSITY OF QUEENSLAND""",null,"""public""","[""fastq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX20720528""","""Australia""","""Oceania""","[""Australia""]",null,"""MinION""","""9419-17ONTBC""","""SINGLE""","""RANDOM""","""METAGENOMIC""",null,88,77,"""Escherichia coli""","""OXFORD_NANOPORE""",2023-06-19 00:00:00 UTC,"""SRS18011968""","""SRP444682""",null,null,null,null,"""[""2021""]""",null,null,null,null,null,null,null,null,"""[""Homo sapiens""]""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""88439849""","""81459369""","""""2023-06-19T19:32:00.000Z""""",null,"""""35766867""""","""other"""
"""P36""",[],"""SRR29161330""","""WGS""",1250,"""PRJNA1116137""","""SAMN41523629""","[""Metagenome or environmental""]","""THE OHIO STATE UNIVERSITY""",2018-04-13,"""public""","[""fastq"", ""sra""]"

In [ ]:
## Filter metadata table

In [16]:
pig_df = metadata2_df.filter(
    (pl.col('bases').cast(pl.Int64) >= 1e9) &
    (pl.col('broad_cat') == 'pig')
)

In [17]:
pig_df

sample_name,sample_name_sam,acc,assay_type,avgspotlen,bioproject,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search,broad_cat
str,list[str],str,str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""JS_LAK4_S129""",[],"""SRR26443177""","""WGS""",299,"""PRJNA688310""","""SAMN37885830""","[""Metagenome or environmental""]","""UNIVERSITY COLLEGE LONDON""",2018-04-01,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX22147725""","""United Kingdom""","""Europe""","[""United Kingdom: Hertfordshire""]",null,"""Illumina HiSeq 1000""","""JS-AP4 1""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,7731,2272,"""pig gut metagenome""","""ILLUMINA""",2023-10-19 00:00:00 UTC,"""SRS19207410""","""SRP467380""",null,null,null,null,"""[""2018-04""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""51.7138 N 0.2103 W""""",null,null,null,null,"""7731646914""","""2382612643""","""""2023-10-19T16:38:00.000Z""""",null,"""""37885830""""","""pig"""
"""215_887_G6_V4""",[],"""SRR25936251""","""WGS""",300,"""PRJNA1010706""","""SAMN37309252""","[""Metagenome or environmental""]","""INSTITUTE OF AGRIFOOD RESERACH…",2020-12-09,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX21655220""","""Spain""","""Europe""","[""Spain: Lleida""]",null,"""Illumina NovaSeq 6000""","""215_887_G6_V4""","""PAIRED""","""RANDOM PCR""","""METAGENOMIC""",null,1293,400,"""pig gut metagenome""","""ILLUMINA""",2024-04-09 00:00:00 UTC,"""SRS18823604""","""SRP458977""",null,null,null,null,"""[""2020-12-09""]""",null,null,null,null,null,null,null,null,"""[""Sus scrofa""]""",null,null,null,null,null,null,null,null,null,"""""41.6178 N 0.6202 E""""",null,null,null,null,"""1293695100""","""419550116""","""""2023-09-07T04:01:00.000Z""""",null,"""""1010706""""","""pig"""
"""Fe24/14208""",[],"""SRR11125486""","""WGS""",302,"""PRJNA526405""","""SAMN11098298""","[""Metagenome or environmental""]","""UNIVERSITY OF TECHNOLOGY SYDNE…",2017-02-24,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX7761821""","""Australia""","""Oceania""","[""Australia: NSW""]",null,"""Illumina NovaSeq 6000""","""plate_3_D12""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1400,397,"""pig gut metagenome""","""ILLUMINA""",2020-02-20 00:00:00 UTC,"""SRS4498273""","""SRP188615""",null,null,null,null,"""[""2017-02-24""]""",null,"""[""Host- gut""]""",null,"""[""Faecal""]""",null,"""[""Faecal""]""",null,null,"""[""Porcine""]""",null,"""""Intestinal Tract""""",null,null,"""[""male""]""",null,null,null,null,"""""34.1273 S 150.7387 E""""",null,null,null,null,"""1400832738""","""416591685""","""""2020-02-20T21:25:00.000Z""""",null,"""""11098298""""","""pig"""


## Check if 3216 is subset...

In [18]:
names_3216_df = (pl.read_csv('../3216.manifest.csv')
                 .with_columns(acc=pl.col('name'))
                 ['acc'].unique().to_frame()
                )
assert len(names_3216_df) == 3216, len(names_3216_df)

In [19]:
assert len(pig_df.join(names_3216_df, on='acc', how='inner')) == 3216

In [15]:
# good!

## Make list of wort sigs

In [20]:
PATH='/group/ctbrowngrp5/wort/wort-sra/sigs/{acc}.sig'

with open('../pig-metag-initial-selection.txt', 'wt') as fp:
    for acc in pig_df['acc']:
        print(PATH.format(acc=acc), file=fp)

## Process results of search



In [60]:
search_df = pl.read_csv('../pig+human.x.pig-metag-initial-selection.csv')

In [61]:
search_df

query_name,query_md5,match_name,containment,intersect_hashes,ksize,scaled,moltype,match_md5,jaccard,max_containment,average_abund,median_abund,std_abund,query_containment_ani,match_containment_ani,average_containment_ani,max_containment_ani,n_weighted_found,total_weighted_hashes
str,str,str,f64,i64,i64,i64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64
"""GCF_000003025.6 pig (Sus scrof…","""2edac3821d65091319e98478f22e94…","""SRR11124652""",0.000017,4,51,10000,"""DNA""","""1d88a72f3ba9a3b344abac7198073e…",0.000015,0.000127,1.25,1.0,0.433013,0.806337,0.83875,0.822544,0.83875,5,75348
"""GCF_000003025.6 pig (Sus scrof…","""2edac3821d65091319e98478f22e94…","""SRR11124838""",0.000013,3,51,10000,"""DNA""","""d3c2a024f3bc5297a55bbeb203f984…",0.000011,0.000085,1.0,1.0,0.0,0.801801,0.832148,0.816974,0.832148,3,77572
"""GCF_000003025.6 pig (Sus scrof…","""2edac3821d65091319e98478f22e94…","""SRR8960115""",0.003876,908,51,10000,"""DNA""","""6e56d987a1eba4a17433092f58a060…",0.003178,0.017338,1.210352,1.0,0.506372,0.896836,0.923571,0.910204,0.923571,1099,198114
"""GCF_000001405.40 human (Homo s…","""4c128af88821ccd01a6f825f4842fe…","""SRR8960115""",0.000007,2,51,10000,"""DNA""","""6e56d987a1eba4a17433092f58a060…",0.000006,0.000038,2.0,2.0,0.0,0.793152,0.819165,0.806159,0.819165,4,198114
"""GCF_000003025.6 pig (Sus scrof…","""2edac3821d65091319e98478f22e94…","""SRR11184014""",0.00035,82,51,10000,"""DNA""","""244ceb8f120e87017ddcff57c2c849…",0.000315,0.003178,1.207317,1.0,0.461646,0.855534,0.89335,0.874442,0.89335,99,87133
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GCF_000001405.40 human (Homo s…","""4c128af88821ccd01a6f825f4842fe…","""SRR17241642""",0.000317,86,51,10000,"""DNA""","""870286465b8c402bc756f390de9e9e…",0.000139,0.000317,1.5,1.0,0.831782,0.853858,0.849649,0.851753,0.853858,129,1198246
"""GCF_000003025.6 pig (Sus scrof…","""2edac3821d65091319e98478f22e94…","""SRR10313340""",0.000154,36,51,10000,"""DNA""","""58a0ed113f562a4ce28b01771cd3e1…",0.000063,0.000154,1.083333,1.0,0.276385,0.841835,0.835757,0.838796,0.841835,39,477589
"""GCF_000001405.40 human (Homo s…","""4c128af88821ccd01a6f825f4842fe…","""SRR10313340""",0.000022,6,51,10000,"""DNA""","""58a0ed113f562a4ce28b01771cd3e1…",0.00001,0.000022,1.0,1.0,0.0,0.810423,0.806904,0.808664,0.810423,6,477589


In [62]:
names_3216_df

acc
str
"""ERR1135222"""
"""SRR11183882"""
"""ERR1135393"""
"""ERR3211939"""
"""ERR1135201"""
…
"""SRR11183931"""
"""ERR8314761"""
"""SRR8960917"""


In [63]:
search_3216_df = search_df.join(names_3216_df, how='semi', left_on='match_name', right_on='acc')

In [66]:
search_3216_df.group_by('query_name').agg(
    pl.col('containment').max()
)

query_name,containment
str,f64
"""GCF_000001405.40 human (Homo s…",0.033424
"""GCF_000003025.6 pig (Sus scrof…",0.28903


In [68]:
metadata2_3216_df = metadata2_df.join(names_3216_df, how='semi', left_on='acc', right_on='acc')

In [73]:
trydatecutoff = metadata2_3216_df.select(pl.col('releasedate').max()).item()

In [79]:
metadata2_df.filter(
    (pl.col('releasedate') <= trydatecutoff) &
    (pl.col('organism') == 'pig gut metagenome' ) &
    (pl.col('bases').cast(pl.Int64) >= 1e9)
)

sample_name,sample_name_sam,acc,assay_type,avgspotlen,bioproject,biosample,biosamplemodel_sam,center_name,collection_date_sam,consent,datastore_filetype,datastore_provider,datastore_region,ena_first_public_run,ena_last_update_run,experiment,geo_loc_name_country_calc,geo_loc_name_country_continent_calc,geo_loc_name_sam,insertsize,instrument,library_name,librarylayout,libraryselection,librarysource,loaddate,mbases,mbytes,organism,platform,releasedate,sample_acc,sra_study,age,altitude,body_habitat,body_product,collection_date,depth,env_biome,env_broad_scale,env_feature,env_local_scale,env_material,env_medium,env_package,host,host_age,host_body_habitat,host_body_product,host_common_name,host_sex,host_subject_id,host_taxid,investigation_type,isolate,lat_lon,project_name,race,sample_type,source_material_id,bases,bytes,run_file_create_date,run_file_version,primary_search,broad_cat
str,list[str],str,str,i64,str,str,list[str],str,date,str,list[str],list[str],list[str],list[str],list[str],str,str,str,list[str],i64,str,str,str,str,str,"datetime[μs, UTC]",i64,i64,str,str,"datetime[μs, UTC]",str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""Fe24/14208""",[],"""SRR11125486""","""WGS""",302,"""PRJNA526405""","""SAMN11098298""","[""Metagenome or environmental""]","""UNIVERSITY OF TECHNOLOGY SYDNE…",2017-02-24,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX7761821""","""Australia""","""Oceania""","[""Australia: NSW""]",null,"""Illumina NovaSeq 6000""","""plate_3_D12""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1400,397,"""pig gut metagenome""","""ILLUMINA""",2020-02-20 00:00:00 UTC,"""SRS4498273""","""SRP188615""",null,null,null,null,"""[""2017-02-24""]""",null,"""[""Host- gut""]""",null,"""[""Faecal""]""",null,"""[""Faecal""]""",null,null,"""[""Porcine""]""",null,"""""Intestinal Tract""""",null,null,"""[""male""]""",null,null,null,null,"""""34.1273 S 150.7387 E""""",null,null,null,null,"""1400832738""","""416591685""","""""2020-02-20T21:25:00.000Z""""",null,"""""11098298""""","""pig"""
"""SAMEA3663214""","[""BHZ-7B""]","""ERR1135387""","""WGS""",195,"""PRJEB11755""","""SAMEA3663214""",[],"""BEIJING GENOME INSTITUTE""",null,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""ena"", ""gs"", … ""s3""]","[""ena"", ""gs.us-east1"", … ""s3.us-east-1""]","[""2016-01-11""]","[""2018-11-16""]","""ERX1214566""","""China""","""Asia""",[],null,"""Illumina HiSeq 2000""","""130421""","""PAIRED""","""other""","""METAGENOMIC""",null,5827,4393,"""pig gut metagenome""","""ILLUMINA""",2016-01-12 00:00:00 UTC,"""ERS970363""","""ERP013165""",null,null,null,null,"""[""2013""]""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""[""metagenome""]""",null,null,"""[""A Catalogue of the pig Gut M…",null,null,null,"""5827806425""","""4607328246""","""""2016-01-13T01:23:00.000Z""""",null,"""""308698""""","""pig"""
"""Fe24/14208""",[],"""SRR11126080""","""WGS""",302,"""PRJNA526405""","""SAMN11098298""","[""Metagenome or environmental""]","""UNIVERSITY OF TECHNOLOGY SYDNE…",2017-02-24,"""public""","[""fastq"", ""run.zq"", ""sra""]","[""gs"", ""ncbi"", ""s3""]","[""gs.us-east1"", ""ncbi.public"", ""s3.us-east-1""]",[],[],"""SRX7762511""","""Australia""","""Oceania""","[""Australia: NSW""]",null,"""Illumina NovaSeq 6000""","""plate_3_D12""","""PAIRED""","""RANDOM""","""METAGENOMIC""",null,1389,404,"""pig gut metagenome""","""ILLUMINA""",2020-02-20 00:00:00 UTC,"""SRS4498273""","""SRP188615""",null,null,null,null,"""[""2017-02-24""]""",null,"""[""Host- gut""]""",null,"""[""Faecal""]""",null,"""[""Faecal""]""",null,null,"""[""Porcine""]""",null,"""""Intestinal Tract""""",null,null,"""[""male""]""",null,null,null,null,"""""34.1273 S 150.7387 E""""",null,null,null,null,"""1389296640""","""424454702""","""""2020-02-20T22:03:00.000Z""""",null,"""""110

In [55]:
combo_df = search_df.filter(pl.col('query_name').str.contains('Sus scrofa')).group_by('match_name').agg(
    best_cont=pl.col('containment').max()
)

In [58]:
combo_df.filter(pl.col('best_cont') < 0.05)['match_name'].n_unique()

4541

In [59]:
combo_df.join(names_3216_df, left_on='match_name', right_on='acc', how='inner')

match_name,best_cont
str,f64
"""SRR8960688""",0.008985
"""SRR10209675""",0.005844
"""ERR1135279""",0.00388
"""SRR14369226""",0.109062
"""ERR3211920""",0.008759
…,…
"""ERR1135330""",0.001084
"""SRR11184013""",0.001187
"""SRR8960521""",0.00012
